Data Loading

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import joblib

import pandas as pd

amazon = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/amazon_products_sales_data_cleaned.csv")

hm = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/handm.csv")

carbon = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/personal_carbon_footprint_behavior.csv")


print(amazon.head())
print(hm.head())
print(carbon.head())

                                       product_title  product_rating  \
0  BOYA BOYALINK 2 Wireless Lavalier Microphone f...             4.6   
1  LISEN USB C to Lightning Cable, 240W 4 in 1 Ch...             4.3   
2  DJI Mic 2 (2 TX + 1 RX + Charging Case), Wirel...             4.6   
3  Apple AirPods Pro 2 Wireless Earbuds, Active N...             4.6   
4  Apple AirTag 4 Pack. Keep Track of and find Yo...             4.8   

   total_reviews  purchased_last_month  discounted_price  original_price  \
0          375.0                 300.0             89.68          159.00   
1         2457.0                6000.0              9.99           15.99   
2         3044.0                2000.0            314.00          349.00   
3        35882.0               10000.0            162.24          162.24   
4        28988.0               10000.0             72.74           72.74   

  is_best_seller is_sponsored             has_coupon buy_box_availability  \
0       No Badge    Sponsored  Sa

Sustainability Score Generation

In [5]:
import numpy as np

amazon['product_rating'] = amazon['product_rating'].fillna(0)

amazon['discount_percentage'] = amazon['discount_percentage'].fillna(0)

amazon['sustainability_tags'] = amazon['sustainability_tags'].fillna('None')

amazon['sustainability_score'] = (amazon['product_rating'] * 15)

amazon.loc[amazon['sustainability_tags']!= 'None','sustainability_score'] += 25

amazon['sustainability_score'] += (amazon['discount_percentage']* 0.2)

amazon['sustainability_score'] = (amazon['sustainability_score'].clip(0,100))

H&M Sustainability Analysis

In [6]:
hm['recycled'] = hm['materials'].str.contains('Recycled',case=False,na=False)

hm['sustainability_score'] = (hm['recycled'].apply(lambda x: 90 if x else 50))

H&M Sustainability Analysis

In [7]:
def recommend_products( category):

    products = amazon[amazon['product_category'] == category]

    products = products.sort_values(by='sustainability_score',ascending=False)

    return products[['product_title','sustainability_score']].head(5)

print(recommend_products( "Phones" ))

                                          product_title  sustainability_score
0     BOYA BOYALINK 2 Wireless Lavalier Microphone f...                 100.0
3823  Casely iPhone 14 Pro Max Case | Fit Check | Gr...                 100.0
4221  UGREEN 30W USB C Charger, Nexode Foldable GaN ...                 100.0
4585  ​​​​PopSockets Phone Grip with Expanding Kicks...                 100.0
1053  ESR for MagSafe Wallet, 5-Card Holder, Magneti...                 100.0


User Carbon Footprint Analysis

In [8]:
avg_carbon = carbon['carbon_footprint_kg'].mean()

print("Average Carbon Footprint:",avg_carbon)

Average Carbon Footprint: 7.980235714285715


train_model.py

In [10]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestRegressor

amazon = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/amazon_products_sales_data_cleaned.csv")

amazon['product_rating'] = (amazon['product_rating'].fillna(0))

amazon['total_reviews'] = (amazon['total_reviews'].fillna(0))

amazon['purchased_last_month'] = (amazon['purchased_last_month'].fillna(0))

amazon['discount_percentage'] = (amazon['discount_percentage'].fillna(0))

amazon['sustainability_tags'] = (amazon['sustainability_tags'].fillna('None'))

amazon['sustainability_score'] = (amazon['product_rating']*15)

amazon.loc[amazon['sustainability_tags']!='None','sustainability_score']+=25

X = amazon[['product_rating','total_reviews','purchased_last_month','discount_percentage']]

y = amazon['sustainability_score']

X_train,X_test,y_train,y_test = (train_test_split(X,y,test_size=0.2,random_state=42))

model = RandomForestRegressor()

model.fit(X_train,y_train)

joblib.dump(model,"sustainability_model.pkl")

['sustainability_model.pkl']

Prediction

In [11]:
import joblib

model = joblib.load("sustainability_model.pkl")

prediction = model.predict([[4.5,2000,500,20]])
print("Predicted Score:",prediction[0])

Predicted Score: 67.75


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


Streamlit App

In [24]:
import joblib

model = joblib.load("sustainability_model.pkl")

# Example values
rating = 4.5
reviews = 2000
purchases = 500
discount = 25

score = model.predict([
    [
        rating,
        reviews,
        purchases,
        discount
    ]
])[0]

print(f"Sustainability Score: {score:.2f}")

if score > 80:
    print("Highly Sustainable Product")
elif score > 60:
    print("Moderately Sustainable Product")
else:
    print("Low Sustainability Product")

Sustainability Score: 68.00
Moderately Sustainable Product


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
